# Fact-Checking Pipeline Demo — GAN Discriminator

This notebook demonstrates the full fact-checking pipeline step by step, using the **GAN discriminator** (BERT-based) as the primary decision signal and **DBpedia Knowledge Base** as secondary verification.

### Pipeline
1. **Triplet Extraction** (spaCy) — Extract (subject, predicate, object) from a claim
2. **GAN Discriminator** (BERT) — Score the triplet as real or fake based on DBpedia knowledge learned during training
3. **Entity Linking** (DBpedia Lookup API) — Map entities to DBpedia URIs
4. **Knowledge Base Query** (SPARQL) — Verify if a direct relation exists in DBpedia
5. **Final Verdict** — Combine GAN score + KB evidence → SUPPORTED / REFUTED / NOT ENOUGH INFO

## Setup

In [3]:
import sys, os
sys.path.insert(0, os.path.join(os.path.abspath('..'), 'src'))

print('Path configured.')

Path configured.


In [4]:
import logging
logging.basicConfig(level=logging.WARNING)

from triplet_extractor import TripletExtractor
from entity_linker import EntityLinker
from knowledge_query import KnowledgeQuery
from gan_model import FactGAN

extractor = TripletExtractor()
linker = EntityLinker()
kb = KnowledgeQuery()

# Load GAN from local model
gan = FactGAN()
gan.load('../models/gan')
gan.discriminator.eval()
print(f'GAN loaded ({sum(p.numel() for p in gan.discriminator.parameters()):,} params)')
print('All components ready.')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2126.30it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GAN loaded (109,679,361 params)
All components ready.


## Pipeline function

Single function that runs all steps and displays the results at each stage.

In [5]:
def run_pipeline(claim: str, gan_high: float = 0.65, gan_low: float = 0.35):
    """Run the full fact-checking pipeline on a claim, printing each step."""

    print('=' * 70)
    print(f'CLAIM: {claim}')
    print('=' * 70)

    # ── Step 1: Triplet Extraction ────────────────────────────────────
    triplets = extractor.extract(claim)
    print(f'\n[Step 1] Triplet Extraction (spaCy)')
    if triplets:
        for s, p, o in triplets:
            print(f'  Subject:   {s}')
            print(f'  Predicate: {p}')
            print(f'  Object:    {o}')
    else:
        print('  No triplets extracted.')
        print(f'\n>>> VERDICT: NOT ENOUGH INFO (no triplets)')
        return

    # ── Step 2: GAN Discriminator ─────────────────────────────────────
    scores = gan.discriminate_triplets(triplets)
    avg_score = scores.mean().item()
    print(f'\n[Step 2] GAN Discriminator (BERT-based)')
    for (s, p, o), score in zip(triplets, scores.squeeze(-1).tolist()):
        bar = '\u2588' * int(score * 20) + '\u2591' * (20 - int(score * 20))
        print(f'  ({s}, {p}, {o})')
        print(f'  Score: {score:.4f}  [{bar}]')
        if score > gan_high:
            print(f'  Interpretation: triplet looks REAL (known fact)')
        elif score < gan_low:
            print(f'  Interpretation: triplet looks FAKE (contradicts known facts)')
        else:
            print(f'  Interpretation: UNCERTAIN (not enough signal)')

    # ── Step 3: Entity Linking ────────────────────────────────────────
    entity_uris = {}
    print(f'\n[Step 3] Entity Linking (DBpedia Lookup API)')
    for s, _, o in triplets:
        for entity in [s, o]:
            if entity not in entity_uris:
                uri = linker.link(entity)
                entity_uris[entity] = uri
                if uri:
                    print(f'  "{entity}" -> {uri.split("/")[-1]}')
                else:
                    print(f'  "{entity}" -> NOT FOUND in DBpedia')

    # ── Step 4: Knowledge Base Query ──────────────────────────────────
    kb_found = False
    print(f'\n[Step 4] Knowledge Base Query (SPARQL)')
    for s, p, o in triplets:
        s_uri = entity_uris.get(s)
        o_uri = entity_uris.get(o)
        if not s_uri and not o_uri:
            print(f'  {s} <-> {o}: SKIPPED (no URIs)')
            continue
        result = kb.verify_triplet(s_uri, o_uri)
        if result['found']:
            kb_found = True
            preds = [pr.split('/')[-1] for pr in result['predicates'][:3]]
            print(f'  {s} <-> {o}: FOUND via {', '.join(preds)}')
        else:
            print(f'  {s} <-> {o}: NOT FOUND')

    # ── Step 5: Final Verdict ─────────────────────────────────────────
    if avg_score > gan_high:
        verdict = 'SUPPORTED'
    elif avg_score < gan_low:
        verdict = 'REFUTED'
    else:
        verdict = 'NOT ENOUGH INFO'

    confidence = avg_score if verdict == 'SUPPORTED' else (1 - avg_score)
    if kb_found and verdict == 'SUPPORTED':
        confidence = min(0.99, confidence * 1.2)
    elif not kb_found and verdict == 'SUPPORTED':
        confidence *= 0.8

    print(f'\n[Step 5] Final Verdict')
    print(f'  GAN score (primary):  {avg_score:.4f}')
    print(f'  KB verification:      {"CONFIRMED" if kb_found else "NO MATCH"}')
    print(f'  ┌─────────────────────────────────────────┐')
    print(f'  │  VERDICT: {verdict:^17s} ({confidence:.0%} conf.)  │')
    print(f'  └─────────────────────────────────────────┘')

    return {
        'claim': claim,
        'triplets': triplets,
        'gan_score': avg_score,
        'kb_found': kb_found,
        'verdict': verdict,
        'confidence': confidence,
    }

## Single claim demo

Change the claim below to test any statement.

In [6]:
result = run_pipeline("Paris is the capital of France")

CLAIM: Paris is the capital of France

[Step 1] Triplet Extraction (spaCy)
  Subject:   Paris
  Predicate: capital
  Object:    France

[Step 2] GAN Discriminator (BERT-based)
  (Paris, capital, France)
  Score: 0.8662  [█████████████████░░░]
  Interpretation: triplet looks REAL (known fact)

[Step 3] Entity Linking (DBpedia Lookup API)
  "Paris" -> Paris
  "France" -> France

[Step 4] Knowledge Base Query (SPARQL)
  Paris <-> France: FOUND via rdf-schema#seeAlso, wikiPageWikiLink, country

[Step 5] Final Verdict
  GAN score (primary):  0.8662
  KB verification:      CONFIRMED
  ┌─────────────────────────────────────────┐
  │  VERDICT:     SUPPORTED     (99% conf.)  │
  └─────────────────────────────────────────┘


## Batch evaluation

Run the pipeline on a set of claims with expected labels to measure accuracy.

In [7]:
test_claims = [
    # (claim, expected verdict)
    ("Paris is the capital of France", "SUPPORTED"),
    ("Rome is the capital of Italy", "SUPPORTED"),
    ("Tokyo is the capital of Japan", "SUPPORTED"),
    ("Albert Einstein developed the theory of relativity", "SUPPORTED"),
    ("The Eiffel Tower is located in Paris", "SUPPORTED"),
    ("London is the capital of France", "REFUTED"),
    ("Berlin is the capital of Italy", "REFUTED"),
    ("Napoleon was born in England", "REFUTED"),
    ("Hitler was a nice person", "REFUTED"),
    ("There is a connection between pizza and quantum physics", "NOT ENOUGH INFO"),
]

results = []
for claim, expected in test_claims:
    r = run_pipeline(claim)
    if r:
        r['expected'] = expected
        results.append(r)
    print()

CLAIM: Paris is the capital of France

[Step 1] Triplet Extraction (spaCy)
  Subject:   Paris
  Predicate: capital
  Object:    France

[Step 2] GAN Discriminator (BERT-based)
  (Paris, capital, France)
  Score: 0.8662  [█████████████████░░░]
  Interpretation: triplet looks REAL (known fact)

[Step 3] Entity Linking (DBpedia Lookup API)
  "Paris" -> Paris
  "France" -> France

[Step 4] Knowledge Base Query (SPARQL)
  Paris <-> France: FOUND via rdf-schema#seeAlso, wikiPageWikiLink, country

[Step 5] Final Verdict
  GAN score (primary):  0.8662
  KB verification:      CONFIRMED
  ┌─────────────────────────────────────────┐
  │  VERDICT:     SUPPORTED     (99% conf.)  │
  └─────────────────────────────────────────┘

CLAIM: Rome is the capital of Italy

[Step 1] Triplet Extraction (spaCy)
  Subject:   Rome
  Predicate: capital
  Object:    Italy

[Step 2] GAN Discriminator (BERT-based)
  (Rome, capital, Italy)
  Score: 0.8804  [█████████████████░░░]
  Interpretation: triplet looks REAL (k

ERROR:entity_linker:DBpedia Lookup failed for 'theory of relativity': HTTPSConnectionPool(host='lookup.dbpedia.org', port=443): Read timed out. (read timeout=15)


  "the theory of relativity" -> Theory_of_relativity

[Step 4] Knowledge Base Query (SPARQL)
  Albert Einstein <-> the theory of relativity: FOUND via rdf-schema#seeAlso, wikiPageWikiLink

[Step 5] Final Verdict
  GAN score (primary):  0.8779
  KB verification:      CONFIRMED
  ┌─────────────────────────────────────────┐
  │  VERDICT:     SUPPORTED     (99% conf.)  │
  └─────────────────────────────────────────┘

CLAIM: The Eiffel Tower is located in Paris

[Step 1] Triplet Extraction (spaCy)
  Subject:   The Eiffel Tower
  Predicate: is located in
  Object:    Paris

[Step 2] GAN Discriminator (BERT-based)
  (The Eiffel Tower, is located in, Paris)
  Score: 0.8674  [█████████████████░░░]
  Interpretation: triplet looks REAL (known fact)

[Step 3] Entity Linking (DBpedia Lookup API)
  "The Eiffel Tower" -> Eiffel_Tower
  "Paris" -> Paris

[Step 4] Knowledge Base Query (SPARQL)
  The Eiffel Tower <-> Paris: FOUND via wikiPageWikiLink, location, owner

[Step 5] Final Verdict
  GAN score 

ERROR:entity_linker:DBpedia Lookup failed for 'London': HTTPSConnectionPool(host='lookup.dbpedia.org', port=443): Read timed out. (read timeout=15)


  "London" -> London
  "France" -> France

[Step 4] Knowledge Base Query (SPARQL)


...

  London <-> France: NOT FOUND

[Step 5] Final Verdict
  GAN score (primary):  0.2841
  KB verification:      NO MATCH
  ┌─────────────────────────────────────────┐
  │  VERDICT:      REFUTED      (72% conf.)  │
  └─────────────────────────────────────────┘

CLAIM: Berlin is the capital of Italy

[Step 1] Triplet Extraction (spaCy)
  Subject:   Berlin
  Predicate: capital
  Object:    Italy

[Step 2] GAN Discriminator (BERT-based)
  (Berlin, capital, Italy)
  Score: 0.3288  [██████░░░░░░░░░░░░░░]
  Interpretation: triplet looks FAKE (contradicts known facts)

[Step 3] Entity Linking (DBpedia Lookup API)
  "Berlin" -> Berlin
  "Italy" -> Italy

[Step 4] Knowledge Base Query (SPARQL)


ERROR:knowledge_query:JSON fetch failed for http://dbpedia.org/resource/Italy: 500 Server Error: SPARQL Request Failed for url: https://dbpedia.org/data/Italy.json


  Berlin <-> Italy: NOT FOUND

[Step 5] Final Verdict
  GAN score (primary):  0.3288
  KB verification:      NO MATCH
  ┌─────────────────────────────────────────┐
  │  VERDICT:      REFUTED      (67% conf.)  │
  └─────────────────────────────────────────┘

CLAIM: Napoleon was born in England

[Step 1] Triplet Extraction (spaCy)
  Subject:   Napoleon
  Predicate: was born in
  Object:    England

[Step 2] GAN Discriminator (BERT-based)
  (Napoleon, was born in, England)
  Score: 0.5334  [██████████░░░░░░░░░░]
  Interpretation: UNCERTAIN (not enough signal)

[Step 3] Entity Linking (DBpedia Lookup API)
  "Napoleon" -> Napoleon
  "England" -> England

[Step 4] Knowledge Base Query (SPARQL)
  Napoleon <-> England: FOUND via wikiPageWikiLink

[Step 5] Final Verdict
  GAN score (primary):  0.5334
  KB verification:      CONFIRMED
  ┌─────────────────────────────────────────┐
  │  VERDICT:  NOT ENOUGH INFO  (47% conf.)  │
  └─────────────────────────────────────────┘

CLAIM: Hitler was a nic

ERROR:entity_linker:DBpedia Lookup failed for 'nice person': HTTPSConnectionPool(host='lookup.dbpedia.org', port=443): Read timed out. (read timeout=15)


  "a nice person" -> NOT FOUND in DBpedia

[Step 4] Knowledge Base Query (SPARQL)
  Hitler <-> a nice person: NOT FOUND

[Step 5] Final Verdict
  GAN score (primary):  0.5060
  KB verification:      NO MATCH
  ┌─────────────────────────────────────────┐
  │  VERDICT:  NOT ENOUGH INFO  (49% conf.)  │
  └─────────────────────────────────────────┘

CLAIM: There is a connection between pizza and quantum physics

[Step 1] Triplet Extraction (spaCy)
  No triplets extracted.

>>> VERDICT: NOT ENOUGH INFO (no triplets)



In [8]:
# Summary table
print(f'{"Claim":50s} {"Expected":18s} {"Predicted":18s} {"GAN":>6s}  Match')
print('-' * 100)

correct = 0
for r in results:
    match = 'OK' if r['expected'] == r['verdict'] else 'MISS'
    if match == 'OK':
        correct += 1
    print(f'{r["claim"]:50s} {r["expected"]:18s} {r["verdict"]:18s} {r["gan_score"]:6.3f}  {match}')

print(f'\nAccuracy: {correct}/{len(results)} ({correct/len(results):.0%})')

Claim                                              Expected           Predicted             GAN  Match
----------------------------------------------------------------------------------------------------
Paris is the capital of France                     SUPPORTED          SUPPORTED           0.866  OK
Rome is the capital of Italy                       SUPPORTED          SUPPORTED           0.880  OK
Tokyo is the capital of Japan                      SUPPORTED          SUPPORTED           0.889  OK
Albert Einstein developed the theory of relativity SUPPORTED          SUPPORTED           0.878  OK
The Eiffel Tower is located in Paris               SUPPORTED          SUPPORTED           0.867  OK
London is the capital of France                    REFUTED            REFUTED             0.284  OK
Berlin is the capital of Italy                     REFUTED            REFUTED             0.329  OK
Napoleon was born in England                       REFUTED            NOT ENOUGH INFO     0.533 

In [9]:
import sys, os
sys.path.insert(0, os.path.join(os.path.abspath('..'), 'src'))

print('Path configured.')
print(f'Added to path: {os.path.join(os.path.abspath(".."), "src")}')
print(f'Path exists: {os.path.exists(os.path.join(os.path.abspath(".."), "src"))}')


Path configured.
Added to path: /Users/marcoserhal/Desktop/fact-checker/src
Path exists: True
